# 🎨 WarnaIn: AI Image Colorization Pipeline (PyTorch + CIE Lab)
Pipeline lengkap untuk melatih model pewarnaan citra (ResNet U-Net) menggunakan dataset COCO 2017 Val (5.000 citra).

In [ ]:
# 1. Setup Lingkungan & Akselerasi GPU
!nvidia-smi
%pip install -q opencv-python pillow matplotlib tqdm scipy scikit-image

In [ ]:
# 2. Download Dataset COCO 2017 Validation Set (5.000 citra, ~800MB)
import os
import urllib.request
import zipfile

os.makedirs('data/raw', exist_ok=True)
zip_path = 'data/raw/val2017.zip'

if not os.path.exists(zip_path):
    print('Mengunduh COCO 2017 val dataset...')
    url = 'http://images.cocodataset.org/zips/val2017.zip'
    urllib.request.urlretrieve(url, zip_path)
    print('Download selesai! Mengekstrak...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('data/raw')
    print('Ekstraksi selesai!')
else:
    print('Dataset zip sudah ada.')

In [ ]:
# 3. Pembagian Dataset (Train 80%, Val 10%, Test 10%)
from src.dataset import prepare_dataset_splits

raw_dir = 'data/raw/val2017' if os.path.exists('data/raw/val2017') else 'data/raw'
summary = prepare_dataset_splits(raw_dir=raw_dir, output_dir='data/splits')
print('Ringkasan Pembagian:', summary)

In [ ]:
# 4. Training Loop Model ResNet-18 U-Net
!python -m src.train --train_dir data/splits/train --val_dir data/splits/val --epochs 25 --batch_size 16 --device cuda

In [ ]:
# 5. Evaluasi Kuantitatif pada Test Set (PSNR, SSIM, MAE)
!python -m src.evaluate --model_path checkpoints/best_model.pth --test_dir data/splits/test --device cuda

In [ ]:
# 6. Visualisasi Hasil Colorization pada Sampel Citra
import torch
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from src.model import build_colorizer
from src.utils import image_to_tensor, tensor_to_rgb

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = build_colorizer('checkpoints/best_model.pth', device=device)

# Ambil satu citra test
test_imgs = [os.path.join('data/splits/test', f) for f in os.listdir('data/splits/test') if f.endswith(('.jpg', '.png'))]
if test_imgs:
    sample_path = test_imgs[0]
    orig = Image.open(sample_path).convert('RGB')
    L, _, _ = image_to_tensor(orig, target_size=(256, 256))
    with torch.no_grad():
        pred_ab = model(L.to(device))
    colorized = tensor_to_rgb(L[0], pred_ab[0])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    ax1.imshow(L.squeeze().cpu().numpy(), cmap='gray')
    ax1.set_title('Input Hitam-Putih (L)')
    ax1.axis('off')
    ax2.imshow(colorized)
    ax2.set_title('Prediksi AI (WarnaIn)')
    ax2.axis('off')
    plt.show()